# 1. Imports, configuration and reproducibility

In [1]:
from pathlib import Path
import copy
import pickle
import random
import time
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

# ============================================================
# Reproducibility
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ============================================================
# Paths
# ============================================================

PROCESSED_DATA_DIR = Path("../Data/Processed")
MODEL_DIR = Path("../Models")

TRAIN_PATH = PROCESSED_DATA_DIR / "train_air_quality.parquet"
VAL_PATH = PROCESSED_DATA_DIR / "validation_air_quality.parquet"
PREPROCESSOR_PATH = PROCESSED_DATA_DIR / "preprocessing_metadata.pkl"

TFT_MODEL_PATH = MODEL_DIR / "tft_model.pth"
PATCHTST_MODEL_PATH = MODEL_DIR / "patchtst_model.pth"
TRAINING_HISTORY_PATH = MODEL_DIR / "training_history.pkl"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# Device
# ============================================================

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

# ============================================================
# Training Configuration
# ============================================================

BATCH_SIZE = 128
NUM_WORKERS = 0

MAX_EPOCHS = 20
PATIENCE = 4

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

HIDDEN_SIZE = 128
DROPOUT = 0.1

PATCH_LENGTH = 24
PATCH_STRIDE = 12

print("=" * 65)
print("MODEL TRAINING CONFIGURATION")
print("=" * 65)

print(f"Device          : {DEVICE}")
print(f"Batch size      : {BATCH_SIZE}")
print(f"Maximum epochs  : {MAX_EPOCHS}")
print(f"Early stopping  : {PATIENCE}")
print(f"Learning rate   : {LEARNING_RATE}")

if DEVICE.type == "mps":
    print("\nApple Silicon MPS acceleration enabled.")

MODEL TRAINING CONFIGURATION
Device          : cuda
Batch size      : 128
Maximum epochs  : 20
Early stopping  : 4
Learning rate   : 0.001


# 2. Load Metadata and Training/validation

In [2]:
# ============================================================
# Load Previous Notebook Outputs
# ============================================================

for path in [
    TRAIN_PATH,
    VAL_PATH,
    PREPROCESSOR_PATH,
]:
    if not path.exists():
        raise FileNotFoundError(
            f"Required file not found: {path}\n"
            "Run 06_data_preprocessing.ipynb first."
        )

with open(PREPROCESSOR_PATH, "rb") as file:
    metadata = pickle.load(file)

ENCODER_LENGTH = metadata["encoder_length"]
PREDICTION_LENGTH = metadata["prediction_length"]

TARGET_COLUMNS = metadata["target_columns"]
NUMERIC_FEATURE_COLUMNS = metadata["numeric_feature_columns"]
NORMALIZATION_COLUMNS = metadata["normalization_columns"]
NORMALIZATION_STATS = metadata["normalization_stats"]

FEATURE_COLUMNS = [
    column
    for column in NUMERIC_FEATURE_COLUMNS
    if column != "time_idx"
]

TARGET_INDICES = [
    FEATURE_COLUMNS.index(column)
    for column in TARGET_COLUMNS
]

NUM_FEATURES = len(FEATURE_COLUMNS)
NUM_TARGETS = len(TARGET_COLUMNS)

# ============================================================
# Load Data
# ============================================================

train_df = pd.read_parquet(TRAIN_PATH)
val_df = pd.read_parquet(VAL_PATH)

for df in [train_df, val_df]:

    df["station_id"] = df["station_id"].astype(str)
    df["sequence_segment_id"] = (
        df["sequence_segment_id"].astype(str)
    )

    df.sort_values(
        [
            "station_id",
            "sequence_segment_id",
            "timestamp",
        ],
        inplace=True,
    )

    df.reset_index(drop=True, inplace=True)

# ============================================================
# Normalize Using Train-Only Statistics
# ============================================================

# ============================================================
# Handle Structural Missing Values
# ============================================================

STRUCTURAL_FILL_VALUES = {
    "hours_since_previous": 0.0,
}

for split_name, df in {
    "Train": train_df,
    "Validation": val_df,
}.items():

    for column, fill_value in STRUCTURAL_FILL_VALUES.items():

        if column in df.columns:

            missing_before = df[column].isna().sum()

            df[column] = df[column].fillna(
                fill_value
            )

            print(
                f"{split_name:<10} | "
                f"{column:<25} | "
                f"Filled: {missing_before:,}"
            )


# ============================================================
# Normalize Using Train-Only Statistics
# ============================================================

for column in NORMALIZATION_COLUMNS:

    stats = NORMALIZATION_STATS[column]

    train_df[column] = (
        (
            train_df[column].astype(np.float32)
            - stats["mean"]
        )
        / stats["std"]
    ).astype(np.float32)

    val_df[column] = (
        (
            val_df[column].astype(np.float32)
            - stats["mean"]
        )
        / stats["std"]
    ).astype(np.float32)


# ============================================================
# Final Numerical Validation
# ============================================================

for split_name, df in {
    "Train": train_df,
    "Validation": val_df,
}.items():

    values = df[
        FEATURE_COLUMNS
    ].to_numpy(dtype=np.float32)

    if not np.isfinite(values).all():

        problematic_columns = []

        for column in FEATURE_COLUMNS:

            column_values = df[
                column
            ].to_numpy(dtype=np.float32)

            if not np.isfinite(
                column_values
            ).all():

                problematic_columns.append(
                    column
                )

        raise ValueError(
            f"{split_name} contains NaN/Inf "
            f"after preprocessing. "
            f"Columns: {problematic_columns}"
        )

print(
    "\nAll training and validation "
    "features are finite."
)

Train      | hours_since_previous      | Filled: 66
Validation | hours_since_previous      | Filled: 0

All training and validation features are finite.


# 3. Memory efficient lazy sliding window dataset

In [3]:
# ============================================================
# Lazy Time-Series Window Dataset
# ============================================================

class AirQualityWindowDataset(Dataset):

    def __init__(
        self,
        dataframe,
        feature_columns,
        target_indices,
        encoder_length,
        prediction_length,
    ):

        self.encoder_length = encoder_length
        self.prediction_length = prediction_length

        self.target_indices = np.asarray(
            target_indices,
            dtype=np.int64,
        )

        self.values = dataframe[
            feature_columns
        ].to_numpy(dtype=np.float32)

        self.window_starts = []

        grouped_indices = dataframe.groupby(
            [
                "station_id",
                "sequence_segment_id",
            ],
            sort=False,
            observed=True,
        ).indices

        required_length = (
            encoder_length + prediction_length
        )

        for indices in grouped_indices.values():

            indices = np.asarray(indices)

            segment_length = len(indices)

            number_of_windows = (
                segment_length
                - required_length
                + 1
            )

            if number_of_windows <= 0:
                continue

            segment_start = int(indices[0])

            self.window_starts.extend(
                range(
                    segment_start,
                    segment_start + number_of_windows,
                )
            )

        self.window_starts = np.asarray(
            self.window_starts,
            dtype=np.int64,
        )

    def __len__(self):

        return len(self.window_starts)

    def __getitem__(self, index):

        start = self.window_starts[index]

        encoder_end = (
            start + self.encoder_length
        )

        prediction_end = (
            encoder_end
            + self.prediction_length
        )

        x = self.values[
            start:encoder_end
        ]

        y = self.values[
            encoder_end:prediction_end,
            self.target_indices,
        ]

        return (
            torch.from_numpy(x),
            torch.from_numpy(y),
        )


# ============================================================
# Create Datasets
# ============================================================

train_dataset = AirQualityWindowDataset(
    dataframe=train_df,
    feature_columns=FEATURE_COLUMNS,
    target_indices=TARGET_INDICES,
    encoder_length=ENCODER_LENGTH,
    prediction_length=PREDICTION_LENGTH,
)

val_dataset = AirQualityWindowDataset(
    dataframe=val_df,
    feature_columns=FEATURE_COLUMNS,
    target_indices=TARGET_INDICES,
    encoder_length=ENCODER_LENGTH,
    prediction_length=PREDICTION_LENGTH,
)

# ============================================================
# Data Loaders
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=False,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=False,
)

# ============================================================
# Validation
# ============================================================

sample_x, sample_y = next(iter(train_loader))

assert sample_x.shape[1:] == (
    ENCODER_LENGTH,
    NUM_FEATURES,
)

assert sample_y.shape[1:] == (
    PREDICTION_LENGTH,
    NUM_TARGETS,
)

print("=" * 65)
print("WINDOW DATASETS CREATED")
print("=" * 65)

print(f"Training windows   : {len(train_dataset):,}")
print(f"Validation windows : {len(val_dataset):,}")

print(f"\nSample X shape : {sample_x.shape}")
print(f"Sample Y shape : {sample_y.shape}")

print("\nLazy window generation enabled.")

WINDOW DATASETS CREATED
Training windows   : 1,559,445
Validation windows : 560,498

Sample X shape : torch.Size([128, 168, 32])
Sample Y shape : torch.Size([128, 15, 9])

Lazy window generation enabled.


# 4. Define TFT and PatchTST Models

In [4]:
# ============================================================
# MPS-Safe TFT-Style Forecasting Model
# ============================================================

class TFTModel(nn.Module):

    def __init__(
        self,
        input_size,
        hidden_size,
        prediction_length,
        num_targets,
        dropout,
    ):

        super().__init__()

        self.prediction_length = prediction_length
        self.num_targets = num_targets

        self.variable_gate = nn.Sequential(
            nn.Linear(input_size, input_size),
            nn.Sigmoid(),
        )

        self.input_projection = nn.Linear(
            input_size,
            hidden_size,
        )

        self.temporal_encoder = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            batch_first=True,
        )

        # MPS-safe: attention dropout must be 0
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=4,
            dropout=0.0,
            batch_first=True,
        )

        self.normalization = nn.LayerNorm(
            hidden_size
        )

        # Regular dropout remains outside attention
        self.dropout = nn.Dropout(dropout)

        self.output_layer = nn.Linear(
            hidden_size,
            prediction_length * num_targets,
        )

    def forward(self, x):

        gate = self.variable_gate(x)

        x = x * gate

        x = self.input_projection(x)

        encoded, _ = self.temporal_encoder(x)

        attended, _ = self.attention(
            encoded,
            encoded,
            encoded,
            need_weights=False,
        )

        encoded = self.normalization(
            encoded + attended
        )

        context = encoded[:, -1]

        context = self.dropout(context)

        output = self.output_layer(context)

        return output.view(
            -1,
            self.prediction_length,
            self.num_targets,
        )


# ============================================================
# MPS-Safe PatchTST-Style Forecasting Model
# ============================================================

class PatchTSTModel(nn.Module):

    def __init__(
        self,
        input_size,
        encoder_length,
        prediction_length,
        num_targets,
        hidden_size,
        patch_length,
        patch_stride,
        dropout,
    ):

        super().__init__()

        self.patch_length = patch_length
        self.patch_stride = patch_stride

        self.prediction_length = prediction_length
        self.num_targets = num_targets

        self.num_patches = (
            (
                encoder_length
                - patch_length
            )
            // patch_stride
        ) + 1

        self.patch_projection = nn.Linear(
            patch_length * input_size,
            hidden_size,
        )

        self.position_embedding = nn.Parameter(
            torch.zeros(
                1,
                self.num_patches,
                hidden_size,
            )
        )

        # MPS-safe Transformer attention
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size,
            nhead=4,
            dim_feedforward=hidden_size * 4,
            dropout=0.0,
            batch_first=True,
            norm_first=True,
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=3,
        )

        self.normalization = nn.LayerNorm(
            hidden_size
        )

        # Regular model dropout outside attention
        self.dropout = nn.Dropout(dropout)

        self.output_layer = nn.Linear(
            hidden_size,
            prediction_length * num_targets,
        )

    def forward(self, x):

        patches = x.unfold(
            dimension=1,
            size=self.patch_length,
            step=self.patch_stride,
        )

        patches = patches.permute(
            0,
            1,
            3,
            2,
        )

        patches = patches.reshape(
            patches.size(0),
            patches.size(1),
            -1,
        )

        patches = self.patch_projection(
            patches
        )

        patches = (
            patches
            + self.position_embedding
        )

        encoded = self.transformer(
            patches
        )

        context = self.normalization(
            encoded.mean(dim=1)
        )

        context = self.dropout(context)

        output = self.output_layer(
            context
        )

        return output.view(
            -1,
            self.prediction_length,
            self.num_targets,
        )


# ============================================================
# Initialize Models
# ============================================================

tft_model = TFTModel(
    input_size=NUM_FEATURES,
    hidden_size=HIDDEN_SIZE,
    prediction_length=PREDICTION_LENGTH,
    num_targets=NUM_TARGETS,
    dropout=DROPOUT,
).to(DEVICE)

patchtst_model = PatchTSTModel(
    input_size=NUM_FEATURES,
    encoder_length=ENCODER_LENGTH,
    prediction_length=PREDICTION_LENGTH,
    num_targets=NUM_TARGETS,
    hidden_size=HIDDEN_SIZE,
    patch_length=PATCH_LENGTH,
    patch_stride=PATCH_STRIDE,
    dropout=DROPOUT,
).to(DEVICE)

# ============================================================
# Parameter Counts
# ============================================================

tft_parameters = sum(
    parameter.numel()
    for parameter in tft_model.parameters()
)

patchtst_parameters = sum(
    parameter.numel()
    for parameter in patchtst_model.parameters()
)

print("=" * 65)
print("MODELS INITIALIZED")
print("=" * 65)

print(
    f"TFT parameters      : "
    f"{tft_parameters:,}"
)

print(
    f"PatchTST parameters : "
    f"{patchtst_parameters:,}"
)

# ============================================================
# MPS Forward-Pass Validation
# ============================================================

tft_model.eval()
patchtst_model.eval()

with torch.no_grad():

    test_x = sample_x[:2].to(
        DEVICE
    )

    tft_output = tft_model(
        test_x
    )

    patchtst_output = patchtst_model(
        test_x
    )

assert tft_output.shape == (
    2,
    PREDICTION_LENGTH,
    NUM_TARGETS,
)

assert patchtst_output.shape == (
    2,
    PREDICTION_LENGTH,
    NUM_TARGETS,
)

assert torch.isfinite(
    tft_output
).all()

assert torch.isfinite(
    patchtst_output
).all()

print(
    f"\nTFT output      : "
    f"{tft_output.shape}"
)

print(
    f"PatchTST output : "
    f"{patchtst_output.shape}"
)

print(
    "\nMPS model forward-pass "
    "validation passed."
)

# Return models to training mode
tft_model.train()
patchtst_model.train()

MODELS INITIALIZED
TFT parameters      : 221,095
PatchTST parameters : 712,583

TFT output      : torch.Size([2, 15, 9])
PatchTST output : torch.Size([2, 15, 9])

MPS model forward-pass validation passed.


PatchTSTModel(
  (patch_projection): Linear(in_features=768, out_features=128, bias=True)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.0, inplace=False)
        (dropout2): Dropout(p=0.0, inplace=False)
      )
    )
  )
  (normalization): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (output_layer): Linear(in_features=128, out_features=135, bias=True)
)

# 5. Training and validation functions

In [5]:
# ============================================================
# Training Utilities
# ============================================================

LOSS_FUNCTION = nn.SmoothL1Loss()


def train_one_epoch(
    model,
    data_loader,
    optimizer,
):

    model.train()

    total_loss = 0.0

    for batch_x, batch_y in data_loader:

        batch_x = batch_x.to(DEVICE)
        batch_y = batch_y.to(DEVICE)

        optimizer.zero_grad(
            set_to_none=True
        )

        predictions = model(batch_x)

        loss = LOSS_FUNCTION(
        predictions,
        batch_y,)

        if not torch.isfinite(loss):

            raise FloatingPointError(
                "Non-finite training loss detected. "
                "Check batch inputs, targets, "
                "model outputs, and learning rate."
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0,
        )

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(data_loader)


@torch.no_grad()
def validate_model(
    model,
    data_loader,
):

    model.eval()

    total_loss = 0.0

    for batch_x, batch_y in data_loader:

        batch_x = batch_x.to(DEVICE)
        batch_y = batch_y.to(DEVICE)

        predictions = model(batch_x)

        loss = LOSS_FUNCTION(
        predictions,
        batch_y,
        )

        if not torch.isfinite(loss):

            raise FloatingPointError(
                "Non-finite validation loss detected."
            )

        total_loss += loss.item()

    return total_loss / len(data_loader)


def train_model(
    model,
    model_name,
    model_path,
):

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=2,
    )

    best_validation_loss = float("inf")
    best_state = None

    epochs_without_improvement = 0

    history = {
        "train_loss": [],
        "validation_loss": [],
    }

    print("\n" + "=" * 65)
    print(f"TRAINING {model_name}")
    print("=" * 65)

    for epoch in range(
        1,
        MAX_EPOCHS + 1,
    ):

        start_time = time.time()

        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
        )

        validation_loss = validate_model(
            model,
            val_loader,
        )

        scheduler.step(validation_loss)

        history["train_loss"].append(
            train_loss
        )

        history["validation_loss"].append(
            validation_loss
        )

        elapsed_time = time.time() - start_time

        print(
            f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
            f"Train: {train_loss:.6f} | "
            f"Val: {validation_loss:.6f} | "
            f"Time: {elapsed_time / 60:.2f} min"
        )

        if (
            validation_loss
            < best_validation_loss
        ):

            best_validation_loss = (
                validation_loss
            )

            best_state = copy.deepcopy(
                model.state_dict()
            )

            epochs_without_improvement = 0

            torch.save(
                {
                    "model_state_dict": best_state,
                    "model_name": model_name,
                    "best_validation_loss": (
                        best_validation_loss
                    ),
                    "feature_columns": FEATURE_COLUMNS,
                    "target_columns": TARGET_COLUMNS,
                    "encoder_length": ENCODER_LENGTH,
                    "prediction_length": PREDICTION_LENGTH,
                },
                model_path,
            )

        else:

            epochs_without_improvement += 1

        if (
            epochs_without_improvement
            >= PATIENCE
        ):

            print(
                "\nEarly stopping triggered."
            )

            break

    model.load_state_dict(best_state)

    print(
        f"\nBest validation loss: "
        f"{best_validation_loss:.6f}"
    )

    print(
        f"Model saved to: {model_path}"
    )

    return history

# 6. Train TFT and PatchTST

In [6]:
# ============================================================
# Train TFT
# ============================================================

tft_history = train_model(
    model=tft_model,
    model_name="TFT",
    model_path=TFT_MODEL_PATH,
)


TRAINING TFT
Epoch 01/20 | Train: 0.100750 | Val: 0.071555 | Time: 1.30 min
Epoch 02/20 | Train: 0.092895 | Val: 0.071234 | Time: 1.28 min
Epoch 03/20 | Train: 0.090016 | Val: 0.072679 | Time: 1.30 min
Epoch 04/20 | Train: 0.087820 | Val: 0.072650 | Time: 1.30 min
Epoch 05/20 | Train: 0.086137 | Val: 0.073898 | Time: 1.30 min
Epoch 06/20 | Train: 0.082278 | Val: 0.074734 | Time: 1.34 min

Early stopping triggered.

Best validation loss: 0.071234
Model saved to: ../Models/tft_model.pth


In [7]:
# Release unused MPS cache before second modle
if DEVICE.type =="mps":
    torch.mps.empty_cache()

In [8]:
# =====================================
# Train PatchTST
# =====================================

patchtst_history = train_model (
    model = patchtst_model,
    model_name="PatchTST",
    model_path=PATCHTST_MODEL_PATH,
)

# Release unused MPS cache before second modle
if DEVICE.type =="mps":
    torch.mps.empty_cache()


TRAINING PatchTST
Epoch 01/20 | Train: 0.103374 | Val: 0.072148 | Time: 1.58 min
Epoch 02/20 | Train: 0.094590 | Val: 0.070433 | Time: 1.60 min
Epoch 03/20 | Train: 0.092022 | Val: 0.071482 | Time: 1.57 min
Epoch 04/20 | Train: 0.089874 | Val: 0.071331 | Time: 1.61 min
Epoch 05/20 | Train: 0.088090 | Val: 0.071473 | Time: 1.61 min
Epoch 06/20 | Train: 0.084489 | Val: 0.071486 | Time: 1.54 min

Early stopping triggered.

Best validation loss: 0.070433
Model saved to: ../Models/patchtst_model.pth


# 7. Save Training History and final validation

In [9]:
# ============================================================
# Save Training History
# ============================================================

training_history = {
    "tft": tft_history,
    "patchtst": patchtst_history,
    "device": str(DEVICE),
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "hidden_size": HIDDEN_SIZE,
    "patch_length": PATCH_LENGTH,
    "patch_stride": PATCH_STRIDE,
}

with open(
    TRAINING_HISTORY_PATH,
    "wb",
) as file:

    pickle.dump(
        training_history,
        file,
        protocol=pickle.HIGHEST_PROTOCOL,
    )

# ============================================================
# Final Validation
# ============================================================

assert TFT_MODEL_PATH.exists()
assert PATCHTST_MODEL_PATH.exists()
assert TRAINING_HISTORY_PATH.exists()

tft_checkpoint = torch.load(
    TFT_MODEL_PATH,
    map_location="cpu",
)

patchtst_checkpoint = torch.load(
    PATCHTST_MODEL_PATH,
    map_location="cpu",
)

print("=" * 65)
print("MODEL TRAINING COMPLETED SUCCESSFULLY")
print("=" * 65)

print(
    f"\nTFT best validation loss      : "
    f"{tft_checkpoint['best_validation_loss']:.6f}"
)

print(
    f"PatchTST best validation loss : "
    f"{patchtst_checkpoint['best_validation_loss']:.6f}"
)

print("\nSaved Models")

print(f"TFT      : {TFT_MODEL_PATH}")
print(f"PatchTST : {PATCHTST_MODEL_PATH}")

print(
    f"History  : "
    f"{TRAINING_HISTORY_PATH}"
)

print("\nForecast Configuration")

print(
    f"Encoder window   : "
    f"{ENCODER_LENGTH} hours"
)

print(
    f"Forecast horizon : "
    f"{PREDICTION_LENGTH} hours"
)

print(
    f"Targets          : "
    f"{NUM_TARGETS}"
)

print(
    f"Training device  : "
    f"{DEVICE}"
)

print(
    "\nReady for model evaluation "
    "and ensemble prediction."
)

MODEL TRAINING COMPLETED SUCCESSFULLY

TFT best validation loss      : 0.071234
PatchTST best validation loss : 0.070433

Saved Models
TFT      : ../Models/tft_model.pth
PatchTST : ../Models/patchtst_model.pth
History  : ../Models/training_history.pkl

Forecast Configuration
Encoder window   : 168 hours
Forecast horizon : 15 hours
Targets          : 9
Training device  : cuda

Ready for model evaluation and ensemble prediction.
